# Python, Лекция 7.

[Оригинал](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/07.%20%D0%98%D1%81%D0%BA%D0%BB%D1%8E%D1%87%D0%B5%D0%BD%D0%B8%D1%8F.%20%D0%9A%D0%BE%D0%BD%D1%82%D0%B5%D0%BA%D1%81%D1%82%D0%BD%D1%8B%D0%B5%20%D0%BC%D0%B5%D0%BD%D0%B5%D0%B4%D0%B6%D0%B5%D1%80%D1%8B/Lecture_7.ipynb) лекции.

Итак, сегодня мы поговорим про **исключения** и **контекстные менеджеры**.

### Исключения

Обработка ошибок — одна из наиболее дискуссионных тем в программировании. Можно сказать, что вокруг подходов к организации работы с ошибками во время выполнения программы формируются "религии" среди программистов.

Давайте рассмотрим ситуацию, в которой будет ошибка из-за попытки открытия несуществующего файла:

In [1]:
with open("non_existent_file.txt", "r") as f:
    ...

FileNotFoundError: [Errno 2] No such file or directory: 'non_existent_file.txt'

В различных языках программирования и парадигмах вопросы обработки ошибок решаются по-разному. Наиболее распространённые подходы включают:

— использование кодов возврата (**return codes**) в языке `C`, когда функция сообщает об ошибке через специальное возвращаемое значение,

— введение специального типа (`Result` в `Rust`, `error` в `Go`, `std::expected` в `C++`) и сопутствующих конструкций для его обработки (например, `match` в `Rust`),

— применение механизма **исключений** (`exceptions`) в таких языках, как `Python`, `Java`, `C++`,

Подход к обработке ошибок оказывает существенное влияние и на стиль программирования (идиомы), и на удобство сопровождения программного кода, и на безопасность исполнения.

Рассмотрим дополнительные примеры возникновений исключений:

In [2]:
1 / 0

ZeroDivisionError: division by zero

In [3]:
int("not a number")

ValueError: invalid literal for int() with base 10: 'not a number'

In [4]:
print(not_defined_variable)

NameError: name 'not_defined_variable' is not defined

In [5]:
[0] * 1_000_000_000_000_000_000

MemoryError: 

В `Python` для перехвата исключений используется конструкция `try...except`:

In [6]:
try:
    ...  # код, который может вызвать исключение
except Exception:
    ...  # обработка исключения

Перехват исключений позволяет:

- Не прерывать работу программы при возникновении ошибки.
- Корректно реагировать на разные типы ошибок.
- Выдавать понятные сообщения, делать повторные попытки и т.д.

К примеру:

In [7]:
try:
    x = 1 / 0
except ZeroDivisionError:
    print("Нельзя делить на ноль!")

Нельзя делить на ноль!


Можно обрабатывать несколько типов исключений:

In [8]:
try:
    s = input("Введите число: ")  # 0
    n = int(s)
    print(10 / n)
except ValueError:
    print("Это не число!")
except ZeroDivisionError:
    print("Вы ввели ноль, деление невозможно.")

Введите число: 0
Вы ввели ноль, деление невозможно.


Можно перехватывать сразу несколько исключений, указав их в круглых скобках в виде кортежа (`tuple`) в блоке `except`:

In [9]:
import typing as tp


def add_numbers(a: tp.Any, b: tp.Any) -> float:
    try:
        return float(a) + float(b)
    except (ValueError, TypeError) as err:
        print("Ошибка значения или типа:", err)
        return None


print(add_numbers("10", "5.5"))
print(add_numbers("abc", "3"))
print(add_numbers([1, 2], "3"))

15.5
Ошибка значения или типа: could not convert string to float: 'abc'
None
Ошибка значения или типа: float() argument must be a string or a real number, not 'list'
None


Иногда бывает удобно перехватить сразу все «обычные» исключения (унаследованные от `Exception`), чтобы вывести ошибку пользователю (например, в пользовательском интерфейсе или логе), не прерывая всю программу аварийно:

In [10]:
# вспомогательный код для следующей ячейки
with open("positive.txt", "w") as f:
    print(5, file=f)


with open("zero.txt", "w") as f:
    print(0, file=f)


with open("text.txt", "w") as f:
    print("text", file=f)

In [11]:
def main() -> None:
    # Попробуйте ввести:
    # - positive.txt
    # - zero.txt
    # - text.txt
    # - non_existent_file.txt
    filename = input("Введите имя файла с числом: ")

    try:
        with open(filename) as f:
            num = int(f.read().strip())
        result = 100 / num
    except Exception as exc:
        print("Что-то пошло не так:", exc)
        return

    print("Результат деления 100 на", num, "равен", result)


if __name__ == "__main__":
    main()

Введите имя файла с числом: positive.txt
Результат деления 100 на 5 равен 20.0


В конструкции `try...except` можно также использовать дополнительные блоки `else` и `finally`, чтобы более гибко управлять поведением программы в зависимости от наличия или отсутствия ошибок.


- Блок `else` срабатывает только если в блоке `try` не возникло исключения.
- Блок `finally` выполняется в любом случае: была ошибка или нет (например, для освобождения ресурсов).


In [12]:
# вспомогательный код для следующей ячейки
with open("data.txt", "w") as f:
    print("I hate exceptions", file=f)

In [13]:
def main() -> None:
    # Попробуйте ввести:
    # - data.txt
    # - non_existent_file.txt
    filename = input()

    try:
        file = open(filename)
    except FileNotFoundError:
        print("Файл не найден.")
    else:
        print("Файл успешно открыт!")
        # Здесь можно безопасно работать с файлом, если открытие прошло без ошибок
        print("Первые 10 символов файла:", file.read(10))
    finally:
        # Этот блок выполнится всегда
        try:
            file.close()
            print("Файл закрыт.")
        except NameError:
            print("Файл не был открыт, закрывать нечего.")


if __name__ == "__main__":
    main()

data.txt
Файл успешно открыт!
Первые 10 символов файла: I hate exc
Файл закрыт.


В стандартной библиотеке Python определено множество видов исключений, структурированных в виде иерархии. Ниже приведена полная иерархия стандартных исключений:

https://docs.python.org/3/library/exceptions.html#exception-hierarchy

```
BaseException
 ├── BaseExceptionGroup
 ├── GeneratorExit
 ├── KeyboardInterrupt
 ├── SystemExit
 └── Exception
      ├── ArithmeticError
      │    ├── FloatingPointError
      │    ├── OverflowError
      │    └── ZeroDivisionError
      ├── AssertionError
      ├── AttributeError
      ├── BufferError
      ├── EOFError
      ├── ExceptionGroup [BaseExceptionGroup]
      ├── ImportError
      │    └── ModuleNotFoundError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── MemoryError
      ├── NameError
      │    └── UnboundLocalError
      ├── OSError
      │    ├── BlockingIOError
      │    ├── ChildProcessError
      │    ├── ConnectionError
      │    │    ├── BrokenPipeError
      │    │    ├── ConnectionAbortedError
      │    │    ├── ConnectionRefusedError
      │    │    └── ConnectionResetError
      │    ├── FileExistsError
      │    ├── FileNotFoundError
      │    ├── InterruptedError
      │    ├── IsADirectoryError
      │    ├── NotADirectoryError
      │    ├── PermissionError
      │    ├── ProcessLookupError
      │    └── TimeoutError
      ├── ReferenceError
      ├── RuntimeError
      │    ├── NotImplementedError
      │    ├── PythonFinalizationError
      │    └── RecursionError
      ├── StopAsyncIteration
      ├── StopIteration
      ├── SyntaxError
      │    └── IndentationError
      │         └── TabError
      ├── SystemError
      ├── TypeError
      ├── ValueError
      │    └── UnicodeError
      │         ├── UnicodeDecodeError
      │         ├── UnicodeEncodeError
      │         └── UnicodeTranslateError
      └── Warning
           ├── BytesWarning
           ├── DeprecationWarning
           ├── EncodingWarning
           ├── FutureWarning
           ├── ImportWarning
           ├── PendingDeprecationWarning
           ├── ResourceWarning
           ├── RuntimeWarning
           ├── SyntaxWarning
           ├── UnicodeWarning
           └── UserWarning
```


`BaseException` — "корень" всей иерархии; все остальные исключения наследуются от этого класса.
Исключения делятся на две больших ветви:

- Исключения, наследуемые от `BaseException` (**но НЕ от `Exception`**), например:
    - `KeyboardInterrupt` — прерывание с клавиатуры (`Ctrl+C`)
    
    - `SystemExit` — выход из интерпретатора
    
    - `GeneratorExit` — завершение генератора (узнаем чуть позже, что такое генераторы и что это за исключение)
    
    Эти обычно не стоит перехватывать в `except Exception`, иначе вы блокируете возможность корректного выхода из программы.

- Все остальные (от `Exception`) — это исключения, появляющиеся в повседневном коде, которые обычно мы и обрабатываем.


`try...except` без уточнения типа **перехватывает все исключения**, то есть любые экземпляры, наследуемые от `BaseException`, включая `SystemExit`, `KeyboardInterrupt`, `GeneratorExit` и даже пользовательские, которые напрямую наследуются от `BaseException`.

In [14]:
try:
    raise KeyboardInterrupt("CTRL+C")
except KeyboardInterrupt | ValueError as exc:  # (KeyboardInterrupt, ValueError)
    print("Перехвачено!", exc)
finally:
    print("b")

b


TypeError: catching classes that do not inherit from BaseException is not allowed

In [15]:
class A:
    def __init__(self, a: int):
        if a < 0:
            raise ValueError("a must be positive")
        self.a = a


try:
    A(-1)
except ValueError as exc:
    print(exc)

a must be positive


Уже отмечалось, что перехват исключений, наследуемых непосредственно от `BaseException`, но не являющихся подклассами `Exception`, (таких как `SystemExit`, `KeyboardInterrupt`, `GeneratorExit` и др.) считается плохой практикой и, как правило, должен избегаться.

Для отлова обычных ошибок рекомендуется использовать конструкцию `except Exception:`, чтобы не нарушать корректную работу интерпретатора `Python`, такую как, например, возможность завершить программу с помощью сочетания клавиш `Ctrl+C (KeyboardInterrupt)`.

Важные группы `Exception`:


- `ArithmeticError`: ошибки арифметики — деление на ноль, переполнение, ошибки с плавающей точкой.
- `LookupError`: для неудачных обращений по индексу (`IndexError`) и ключу (`KeyError`).
- `OSError`: все ошибки, связанные с операционной системой — файлы, процессы, соединения.
- `RuntimeError`, `ValueError`, `TypeError` — "общие" ошибки, возникающие при неправильных входных данных, нарушениях логики и т.п.
- `ImportError`, `ModuleNotFoundError`: ошибки импорта модулей.
- `SyntaxError`, `IndentationError`, `TabError`: нарушения синтаксиса, проблемы с отступами в исходном коде.


![image](https://raw.githubusercontent.com/DanielShinoda/ami_python_25_lectures/1897e181ecdaaf7e92f3b7b94bb7a354480a4037/lectures/07.%20%D0%98%D1%81%D0%BA%D0%BB%D1%8E%D1%87%D0%B5%D0%BD%D0%B8%D1%8F.%20%D0%9A%D0%BE%D0%BD%D1%82%D0%B5%D0%BA%D1%81%D1%82%D0%BD%D1%8B%D0%B5%20%D0%BC%D0%B5%D0%BD%D0%B5%D0%B4%D0%B6%D0%B5%D1%80%D1%8B/1f11cc1d97095debe72a443bc576e2ed.jpg)

`IndentationError` — это разновидность `SyntaxError` и относится к исключениям уровня компиляции, возникает во время разбора (парсинга) кода `Python`, до его фактического выполнения. Поэтому перехватить `IndentationError` в обычном `try...except` вокруг кода с ошибкой невозможно — такой код даже не исполнится.

In [16]:
def test():
print("Ошибка отступа!")

IndentationError: expected an indented block after function definition on line 1 (2044899705.py, line 2)

Однако можно обработать ситуацию, когда вы исполняете или компилируете строку с ошибочной программой во время выполнения. Например, с помощью функции `exec()`:

In [17]:
code = """
def test():
print("Ошибка отступа!")
"""

try:
    # функция exec выполняет код, представленный в виде строки
    exec(code)
except IndentationError as e:
    print("IndentationError:", e)
except SyntaxError as e:
    print("Другая синтаксическая ошибка:", e)

IndentationError: expected an indented block after function definition on line 2 (<string>, line 3)


В представленной иерархии существует, пожалуй, наиболее необычный (на взгляд автора) подкласс исключений — `Warning`.

`Warning` и подклассы используются не для ошибок, а для сообщений-предупреждений (не прерывают выполнение, можно перехватывать через модуль `warnings`). Подробное рассмотрение предупреждений будет проведено в заключительной части данного топика.



Отдельно следует отметить, что начиная с `Python 3.11` появились **exception-группы** (`ExceptionGroup`), предназначенные для поддержки работы с несколькими одновременно возникшими исключениями, что особенно актуально при работе с асинхронным кодом. Для обработки таких групп введена специальная конструкция `except*`. Однако в рамках данного материала мы не будем подробно рассматривать этот механизм, поскольку для его понимания требуется предварительное освоение принципов работы с корутинами.



Теперь, когда мы разобрались с перехватом исключений, рассмотрим, как инициировать их возникновение самостоятельно (так называемое "бросание" исключений):

In [18]:
raise Exception("Исключение!")

Exception: Исключение!

Оператор `raise` требует, чтобы в качестве аргумента ему передавался объект, являющийся экземпляром класса, производного от `BaseException`. Передача в качестве аргумента объекта, не являющегося экземпляром класса, производного от `BaseException` (например, строки или числа), приведёт к возникновению исключения `TypeError`:

In [19]:
raise 1

TypeError: exceptions must derive from BaseException

In [20]:
# здесь все ок
a = ValueError("value error")

raise a

ValueError: value error

А ещё можно использовать `raise` внутри блока `except` без указания типа исключения — это позволяет повторно пробросить текущее исключение после выполнения необходимой дополнительной обработки, например, логирования или очистки ресурсов:

In [21]:
try:
    try:
        1 / 0
    except ZeroDivisionError as e:
        print("Логируем ошибку и пробрасываем дальше")
        raise ValueError("a") from e
except ValueError as err:
    print("c")
    raise ValueError("b") from err

Логируем ошибку и пробрасываем дальше
c


ValueError: b

Ещё одной важной возможностью является использование конструкции `raise ... from ...` С её помощью можно явно указать причинно-следственную связь между двумя исключениями: если одно исключение возникает в процессе обработки другого, новое исключение может быть выброшено с ссылкой на исходное. Такой приём называется **цепочкой исключений** (`exception chaining`) и облегчает отладку и анализ ошибок, позволяя видеть, какое исключение послужило причиной нового.

In [22]:
try:
    int("abc")
except ValueError as e:
    raise RuntimeError("Ошибка преобразования данных") from e

RuntimeError: Ошибка преобразования данных

В некоторых случаях стандартных исключений Python бывает недостаточно для выражения специфических ошибок, характерных именно для вашей программы или библиотеки. В таких ситуациях рекомендуется определять собственные типы исключений:

In [23]:
class MyCustomError(Exception):
    """Описание пользовательской ошибки"""

    ...


# Использование собственного исключения:
raise MyCustomError("Что-то пошло не так")

MyCustomError: Что-то пошло не так

Рекомендуется давать пользовательским классам исключений имена, оканчивающиеся на `Error` ([pep8](https://peps.python.org/pep-0008/#exception-names)).

In [24]:
class MySubCustomError(MyCustomError):
    def __str__(self):
        # self.args - это аргументы, которые мы передали в __init__
        return f"Some information of my sub custom error: {self.args}"


try:
    raise MySubCustomError(1)
except MyCustomError as e:
    print(f"Что-то не так: {e=}")
    raise

Что-то не так: e=MySubCustomError(1)


MySubCustomError: Some information of my sub custom error: (1,)

В `Python` также существует специализированный модуль `traceback`, предназначенный для получения и обработки текстового представления трассировки стека (`stack trace`). Подробное знакомство с возможностями этого модуля предлагается провести в рамках практических занятий.


Также на семинарских занятиях целесообразно более подробно рассмотреть использование специальных атрибутов исключений: `__cause__`, `__context__`, `__traceback__`.

Остаётся рассмотреть вопросы, связанные с обработкой предупреждений (`warning`) в Python:

Повторимся. В `Python` предупреждения (`Warning`) — это не исключения, они не останавливают программу, но позволяют информировать программиста или пользователя о потенциальных проблемах. Используются через стандартный модуль `warnings`.

In [25]:
import warnings


warnings.warn("Это предупреждение!")
print(1)

1


/tmp/ipykernel_1031/2918491857.py:4: UserWarning: Это предупреждение!
  warnings.warn("Это предупреждение!")


Примеры разных типов `warning`:

`UserWarning` — базовое, "пользовательское" предупреждение (по умолчанию):

In [26]:
warnings.warn("Обычное пользовательское предупреждение!")

/tmp/ipykernel_1031/860540932.py:1: UserWarning: Обычное пользовательское предупреждение!
  warnings.warn("Обычное пользовательское предупреждение!")


`DeprecationWarning` — предупреждение об устаревшем функционале:

In [27]:
warnings.warn("Этот метод устарел и будет удалён.", DeprecationWarning)

/tmp/ipykernel_1031/1390018883.py:1: DeprecationWarning: Этот метод устарел и будет удалён.
  warnings.warn("Этот метод устарел и будет удалён.", DeprecationWarning)


`SyntaxWarning` — замечание по поводу опасного синтаксиса (чаще всего выводится самим интерпретатором):

In [28]:
x = "hello"
if x is "hello":
    print("Строки совпали")

Строки совпали


<>:2: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
<>:2: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
/tmp/ipykernel_1031/2841816111.py:2: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if x is "hello":


`BytesWarning` — предупреждение работы со строками и байтами

In [29]:
a = "text"
b = b"text"

print(
    a == b
)  # В обычном режиме — False, но если запустить с python -b, будет BytesWarning
# -b     : issue warnings about converting bytes/bytearray to str and comparing
#          bytes/bytearray with str or bytes with int. (-bb: issue errors)

False


`RuntimeWarning` — предупреждение, связанное с логикой выполнения

`FutureWarning` — то, что будет изменено в будущем

`ResourceWarning` — предупреждения об утечках ресурсов

Модуль `warnings` не вызывает исключение, но можно изменить поведение, чтобы, например, превратить предупреждение в ошибку:

In [30]:
warnings.simplefilter("error")
try:
    warnings.warn("Это уже ошибка!", UserWarning)
except UserWarning as e:
    warnings.simplefilter("default")  # !!!
    print("Поймали как ошибку:", e)

Поймали как ошибку: Это уже ошибка!


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Возвращаем режим по умолчанию:

In [31]:
warnings.simplefilter("default")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Можно создать свой подкласс `warning`:

In [32]:
class MyWarning(Warning):
    pass


warnings.warn("Специальное предупреждение!", MyWarning)


/tmp/ipykernel_1031/2076636879.py:5: MyWarning: Специальное предупреждение!
  warnings.warn("Специальное предупреждение!", MyWarning)


Предупреждения (`warning`) в `Python` могут быть программно подавлены или отключены.
На практике это часто делается при помощи модуля `warnings`, чтобы игнорировать нежелательные предупреждения в ходе выполнения кода. Например, при выполнении лабораторных работ по линейной алгебре или геометрии в начале программы вы возможно встретите следующую строку:

In [33]:
import warnings

warnings.filterwarnings("ignore")

In [34]:
warnings.warn("Это предупреждение!")

In [35]:
warnings.simplefilter("default")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Игнорировать предупреждения (`warnings`) —  плохая практика. Лучше разбираться в их причинах и фиксить, либо подавлять выборочно осознанно.

Игнорировать только определенный класс предупреждений:

In [36]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

С определенным текстом:

In [37]:
import warnings

warnings.filterwarnings("ignore", message=".*устарел.*")

Только в определенном модуле:

In [38]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="my_module")

Игнорировать только внутри блока (самый лучший подход, если уж нужно заигнорировать):

In [39]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=UserWarning)
    warnings.warn(
        "Это пользовательское предупреждение!", UserWarning
    )  # Не будет отображено

warnings.warn("Это пользовательское предупреждение!", UserWarning)  # Будет отображено

/tmp/ipykernel_1031/889226964.py:9: UserWarning: Это пользовательское предупреждение!
  warnings.warn("Это пользовательское предупреждение!", UserWarning)  # Будет отображено


### Контекстные менеджеры

Не следует пугаться термина "контекстный менеджер" — вы уже сталкивались с их использованием:

In [40]:
with open("file.txt", "w") as f:
    ...

Контекстные менеджеры в Python обеспечивают автоматическое выполнение определённых действий при входе в блок кода и выходе из него. Cпособ их использования — конструкция `with`.

В приведённом выше примере при входе в блок осуществляется открытие файла для чтения, а при выходе из блока файл автоматически закрывается.

Функция open возвращает объект, реализующий протокол контекстного менеджера, то есть содержащий специальные методы:
- `__enter__` в данном случае возвращает сам объект файла (возвращает `self`)
- `__exit__` обеспечивает автоматический вызов метода `close` для закрытия файла при выходе из блока управления контекстом.

Перейдем к определению. Контекстные менеджеры --- это объекты, реализующие протокол контекстного менеджера (методы `__enter__` и `__exit__`).

Рассмотрим вопрос о назначении контекстных менеджеров. Для чего они необходимы? Контекстные менеджеры гарантируют нам, что определённое действие будет выполнено вне зависимости от того, возникло ли в процессе работы программы исключение или нет:


Без использования контекстных менеджеров обеспечение корректного освобождения ресурсов требует явной обработки, что может привести к ошибкам или утечкам ресурсов. Рассмотрим следующий пример:

In [41]:
def process_file(f):
    # Какая-то потенциально опасная операция с файлом
    f.write("Начинается запись данных...\n")
    raise RuntimeError("Возникла критическая ошибка при обработке файла!")


f = open("file.txt", "w")
try:
    process_file(f)
except Exception as exc:
    print(f"Произошло исключение: {exc}")
finally:
    print("Закрываем файл.")
    try:
        f.close()
        print("Файл успешно закрыт.")
    except Exception as close_exc:
        print(f"Ошибка при закрытии файла: {close_exc}")

Произошло исключение: Возникла критическая ошибка при обработке файла!
Закрываем файл.
Файл успешно закрыт.


В данном случае необходимо явно вызывать метод `close()` для закрытия файла, даже если в процессе работы возникает исключение.

Контекстные менеджеры предоставляют удобный и безопасный механизм для инициализации и гарантированного завершения работы с ресурсом (финализации "контекста"), устраняя необходимость ручного контроля освобождения ресурсов и снижая риск возникновения ошибок:

In [42]:
with open("file.txt", "w") as f:
    process_file(f)

RuntimeError: Возникла критическая ошибка при обработке файла!

Контекстные менеджеры в `Python` реализуют идиому **RAII** (англ. Resource Acquisition Is Initialization) — программный подход, согласно которому инициализация объекта сопровождается захватом (выделением) необходимого ресурса, а освобождение ресурса автоматически происходит при уничтожении объекта. В данном случае получение ресурса осуществимо только через его инициализацию, а освобождение — неразрывно связано с завершением жизни объекта.


Полная демонстрация данной идиомы более характерна для языков программирования, таких как `C++`, где освобождение ресурсов жестко связано с областью видимости объектов и временем их жизни. Подробную работу данной идиомы вы сможете изучить позднее при знакомстве с `C++`.

Приведу несколько примеров использования контекстных менеджеров из стандартной библиотеки `Python` (в основном из модуля `contextlib`). Здесь не будет примеров работы с соединениями к базам данных, так как предполагаю, что большинство с ними пока не сталкивались. Тем не менее, при необходимости подобные примеры легко найти — можете погуглить или попросить какую-нибудь нейросеть.

Также выше был описан пример использования контекстного менеджера, где мы игнорировали определённое предупреждение (`warning`).

In [43]:
import tempfile

# Создание временного файла
with tempfile.TemporaryFile(mode="w+t") as temp:
    temp.write("Temporary content")
    temp.seek(0)
    print(temp.read())

# Файл автоматически удаляется

Temporary content


In [44]:
import contextlib

with open("file.txt", "w") as f:
    with contextlib.redirect_stdout(f):
        print("Этот текст попадет в file.txt, а не на экран")

In [45]:
import contextlib

with contextlib.suppress(FileNotFoundError):
    with open("no_such_file.txt") as f:
        print(f.read())

# Исключение FileNotFoundError будет проигнорировано

In [46]:
import os
import contextlib

# с python 3.11
with contextlib.chdir("/tmp"):
    print(os.getcwd())

# После выхода вернется в исходную директорию

/tmp


Давайте рассмотрим пример собственного контекстного менеджера, который будет измерять время выполнения кода внутри блока `with`. Для этого можно реализовать класс с методами `__enter__` и `__exit__`:

In [47]:
import time
from types import TracebackType


class Timer:
    def __enter__(self) -> "Timer":
        self._start_time: float = time.time()
        print("Выполнение началось.")
        return self

    def __exit__(
        self,
        exctype: type[BaseException] | None,
        excinst: BaseException | None,
        exctb: TracebackType | None,
    ) -> bool | None:
        self._end_time: float = time.time()
        duration: float = self._end_time - self._start_time
        print(f"Выполнение завершено. Прошло времени: {duration:.4f} секунд.")

Давайте разберем аргументы `__exit__`:

- `exctype: type[BaseException] | None` - это любой тип исключения, а `| None` допускает ситуацию без исключения.

- `excinst: BaseException | None` - экземпляр возникшего исключения, если исключения не было, будет передано `None`.

- `exctb: TracebackType | None` - трейсбэк (объект, описывающий стек вызовов в момент возникновения исключения), если исключение не возникло, тут будет `None`.

Возвращаемое значение `bool | None`:
- если метод `__exit__` возвращает `True`, то возникшее исключение считается обработанным и не будет передано дальше.
- если возвращается `False` или `None`, исключение будет проброшено дальше.

In [48]:
with Timer():
    time.sleep(2)

Выполнение началось.
Выполнение завершено. Прошло времени: 2.0002 секунд.


Давайте сделаем еще один пример, в котором поработаем с аргументами `__exit__`:

In [49]:
import traceback
from types import TracebackType


class ExceptionLogger:
    def __init__(self, raise_exception: bool = False):
        self._raise_exception = raise_exception

    def __enter__(self) -> "ExceptionLogger":
        print("Вход в контекстный менеджер")
        return self

    def __exit__(
        self,
        exctype: type[BaseException] | None,
        excinst: BaseException | None,
        exctb: TracebackType | None,
    ) -> None:
        if exctype is not None:
            print("Обнаружено исключение:")
            print(f"Тип: {exctype.__name__}")
            print(f"Сообщение: {excinst}")
            print("Traceback:")
            traceback.print_tb(exctb)
        else:
            print("Выход из контекстного менеджера без исключений.")
        return not self._raise_exception

In [50]:
with ExceptionLogger():
    print("До исключения")
    1 / 0
    print("Этот код не выполнится")

Вход в контекстный менеджер
До исключения
Обнаружено исключение:
Тип: ZeroDivisionError
Сообщение: division by zero
Traceback:


  File "/tmp/ipykernel_1031/1998374206.py", line 3, in <cell line: 0>
    1 / 0
    ~~^~~


In [51]:
with ExceptionLogger(raise_exception=True):
    print("До исключения")
    1 / 0
    print("Этот код не выполнится")

Вход в контекстный менеджер
До исключения
Обнаружено исключение:
Тип: ZeroDivisionError
Сообщение: division by zero
Traceback:


  File "/tmp/ipykernel_1031/3045196034.py", line 3, in <cell line: 0>
    1 / 0
    ~~^~~


ZeroDivisionError: division by zero

In [52]:
with ExceptionLogger(raise_exception=True):
    print("Этот код выполнится")

Вход в контекстный менеджер
Этот код выполнится
Выход из контекстного менеджера без исключений.


Существует более удобный способ создания собственных контекстных менеджеров — с помощью декоратора `@contextmanager` из модуля `contextlib`. Однако для его понимания необходимо предварительно познакомиться с такими концепциями, как генераторы и декораторы. Эти темы будут подробно рассмотрены на последующих лекциях, и тогда мы разберём альтернативный, более лаконичный подход к написанию контекстных менеджеров.

In [53]:
import sys


def extract_traceback_info():
    def inner_function():
        try:
            1 / 0
        except:
            # Extract detailed frame information
            tb = sys.exc_info()[2]

            print("=== extract_tb() ===")
            extracted = traceback.extract_tb(tb)
            for frame_summary in extracted:
                print(f"File: {frame_summary.filename}")
                print(f"Line: {frame_summary.lineno}")
                print(f"Function: {frame_summary.name}")
                print(f"Code: {frame_summary.line}")
                print("---")

            print("=== extract_stack() ===")
            stack_extracted = traceback.extract_stack()
            for frame in stack_extracted[:-5]:  # Skip the last few internal frames
                print(f"{frame.filename}:{frame.lineno} in {frame.name}")

    inner_function()


extract_traceback_info()

=== extract_tb() ===
File: /tmp/ipykernel_1031/1073956921.py
Line: 7
Function: inner_function
Code: 1 / 0
---
=== extract_stack() ===
<frozen runpy>:203 in _run_module_as_main
<frozen runpy>:88 in _run_code
/usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py:37 in <module>
/usr/local/lib/python3.13/dist-packages/traitlets/config/application.py:992 in launch_instance
/usr/local/lib/python3.13/dist-packages/ipykernel/kernelapp.py:712 in start
/usr/local/lib/python3.13/dist-packages/tornado/platform/asyncio.py:211 in start
/usr/lib/python3.13/asyncio/base_events.py:684 in run_forever
/usr/lib/python3.13/asyncio/base_events.py:2061 in _run_once
/usr/lib/python3.13/asyncio/events.py:89 in _run
/usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py:510 in dispatch_queue
/usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py:499 in process_one
/usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py:406 in dispatch_shell
/usr/local/lib/python3.13/dist-pac